# Step 3: Strategy Implementation & Order Execution Backtesting

## 1. Context & Objective
Simulate and validate the trading strategies against real Level-2 order book depth.

In 5-minute binary prediction markets, unrealistic assumptions (like filling large sizes at the midpoint price without slippage) lead to false backtest results. This notebook implements:
1. **Realistic L2 Execution Simulator**: Walks actual top-5 bids and asks to determine true fill prices, volume-weighted slippage, and liquidity exhaustion.
2. **Event-Driven Strategy Engines**:
   - **Strategy 1: Late-Entry Strategy**: Enters late in the market window under low volatility and high directional cushion, protected by dynamic stop-loss suppression.
   - **Strategy 2: Limit Order Simulation Strategy**: Baseline limit-order simulation with discount entry at 0.49, take-profit at 0.70, and emergency exits before expiry.
3. **Settlement & PnL Accounting**: Integrates official Chainlink settlement prices ($1.00$ payout for winning side, $0.00$ for losing side) to compute net dollar return and risk metrics.

## 2. Setup & Dependencies

In [25]:
import pandas as pd
import numpy as np
import json
import requests
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Any

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

try:
    from IPython.display import display
except ImportError:
    display = print

# Optional visualization libraries
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = (10, 5)
except ImportError:
    HAS_PLOTS = False

PROCESSED_DATA_DIR = Path('../data/market_data/processed')
RAW_DATA_DIR = Path('../data/market_data/raw')

# Filter for per-slot feature files (excluding the merged all_features.csv)
feature_files = sorted([f for f in PROCESSED_DATA_DIR.rglob('*.csv') if f.name != 'all_features.csv'])
print(f'Found {len(feature_files)} individual market slot files:')
for f in feature_files:
    print(f'  - {f.name}')

Found 8 individual market slot files:
  - btc-updown-15m-1793942100_features.csv
  - btc-updown-5m-1793932800_features.csv
  - btc-updown-5m-1793933400_features.csv
  - btc-updown-5m-1793940600_features.csv
  - btc-updown-5m-1793941200_features.csv
  - btc-updown-5m-1793941500_features.csv
  - btc-updown-5m-1793941800_features.csv
  - btc-updown-5m-1794454500_features.csv


## 3. Load Feature Datasets

Load the streamlined snapshots and parse the JSON-encoded Level-2 order book depth columns (`up_top5_bids`, `up_top5_asks`, `down_top5_bids`, `down_top5_asks`).

In [26]:
def load_slot_data(file_path: Path) -> pd.DataFrame:
    """Loads a processed feature CSV and parses L2 depth columns."""
    df = pd.read_csv(file_path)
    df['current_time'] = pd.to_datetime(df['current_time'])
    
    def parse_depth(val):
        if isinstance(val, list):
            return val
        if pd.isna(val) or val == '':
            return []
        try:
            return json.loads(val)
        except Exception:
            return []
            
    for col in ['up_top5_bids', 'up_top5_asks', 'down_top5_bids', 'down_top5_asks']:
        if col in df.columns:
            df[col] = df[col].apply(parse_depth)
            
    return df

slots_data = {}
for f in feature_files:
    slug = f.stem.replace('_features', '')
    slots_data[slug] = load_slot_data(f)
    print(f'Loaded {slug}: {len(slots_data[slug])} rows')

# Display sample row
sample_slug = list(slots_data.keys())[0]
print(f'\nSample metadata for {sample_slug}:')
display_cols = ['current_time', 'elapsed_seconds', 'remaining_seconds', 'price_gap', 'rsi_gap', 'atr_btc', 'gap_safety', 'up_midpoint']
display(slots_data[sample_slug][display_cols].head(3))

Loaded btc-updown-15m-1793942100: 300 rows
Loaded btc-updown-5m-1793932800: 15 rows
Loaded btc-updown-5m-1793933400: 16 rows
Loaded btc-updown-5m-1793940600: 15 rows
Loaded btc-updown-5m-1793941200: 15 rows
Loaded btc-updown-5m-1793941500: 15 rows
Loaded btc-updown-5m-1793941800: 300 rows
Loaded btc-updown-5m-1794454500: 50 rows

Sample metadata for btc-updown-15m-1793942100:


,current_time,elapsed_seconds,remaining_seconds,price_gap,rsi_gap,atr_btc,gap_safety,up_midpoint
0,2026-11-06 05:18:58.237321+00:00,238.24,61.76,304.729311,50.0,NaN,NaN,0.925
1,2026-11-06 05:18:59.223518+00:00,239.22,60.78,304.729311,50.0,NaN,NaN,0.925
2,2026-11-06 05:19:00.239039+00:00,240.24,59.76,304.739311,50.0,NaN,NaN,0.925


## 4. Realistic L2 Order Book Execution Simulator

Simulates order placement by walking the actual top-5 depth levels:
- **Buys**: Walk ask levels ascending by price. Realized price is volume-weighted average price (VWAP) of matched liquidity.
- **Sells**: Walk bid levels descending by price to calculate accurate proceeds and slippage.

In [27]:
class L2OrderBookSimulator:
    """
    Simulates realistic order fills against top-5 L2 order book levels.
    """
    @staticmethod
    def simulate_buy(top_asks: List[List[float]], shares_requested: float, limit_price: float) -> Dict[str, Any]:
        if not top_asks:
            return {'filled_shares': 0.0, 'avg_fill_price': 0.0, 'cost': 0.0, 'status': 'UNFILLED'}
            
        remaining_shares = shares_requested
        total_cost = 0.0
        filled_shares = 0.0
        
        for level in top_asks:
            price, size = float(level[0]), float(level[1])
            if price > limit_price:
                break
                
            fill_size = min(remaining_shares, size)
            total_cost += fill_size * price
            filled_shares += fill_size
            remaining_shares -= fill_size
            
            if remaining_shares <= 1e-6:
                break
                
        avg_price = (total_cost / filled_shares) if filled_shares > 0 else 0.0
        status = 'FILLED' if remaining_shares <= 1e-6 else ('PARTIAL' if filled_shares > 0 else 'UNFILLED')
        
        return {
            'filled_shares': round(filled_shares, 4),
            'avg_fill_price': round(avg_price, 4),
            'cost': round(total_cost, 4),
            'status': status
        }

    @staticmethod
    def simulate_sell(top_bids: List[List[float]], shares_requested: float, limit_price: float) -> Dict[str, Any]:
        if not top_bids:
            return {'filled_shares': 0.0, 'avg_fill_price': 0.0, 'proceeds': 0.0, 'status': 'UNFILLED'}
            
        remaining_shares = shares_requested
        total_proceeds = 0.0
        filled_shares = 0.0
        
        for level in top_bids:
            price, size = float(level[0]), float(level[1])
            if price < limit_price:
                break
                
            fill_size = min(remaining_shares, size)
            total_proceeds += fill_size * price
            filled_shares += fill_size
            remaining_shares -= fill_size
            
            if remaining_shares <= 1e-6:
                break
                
        avg_price = (total_proceeds / filled_shares) if filled_shares > 0 else 0.0
        status = 'FILLED' if remaining_shares <= 1e-6 else ('PARTIAL' if filled_shares > 0 else 'UNFILLED')
        
        return {
            'filled_shares': round(filled_shares, 4),
            'avg_fill_price': round(avg_price, 4),
            'proceeds': round(total_proceeds, 4),
            'status': status
        }

print('L2 Execution Simulator ready.')

L2 Execution Simulator ready.


## 5. Settlement Oracle Resolution

Fetches official Chainlink settlement outcomes (`openPrice` vs `closePrice`):
- If `closePrice >= openPrice` $\rightarrow$ **UP** wins ($1.00$ payout per share).
- If `closePrice < openPrice` $\rightarrow$ **DOWN** wins ($1.00$ payout per share).

In [28]:
def fetch_market_resolution(slot_start: int, slot_end: int, symbol: str = 'BTC') -> Dict[str, Any]:
    url = 'https://polymarket.com/api/crypto/crypto-price'
    params = {
        'symbol': symbol.upper(),
        'variant': 'fiveminute',
        'eventStartTime': str(slot_start),
        'endDate': str(slot_end)
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            open_p = data.get('openPrice')
            close_p = data.get('closePrice')
            completed = data.get('completed', False)
            winner = ('UP' if close_p >= open_p else 'DOWN') if (completed and open_p and close_p) else None
            return {
                'open_price': open_p,
                'close_price': close_p,
                'completed': completed,
                'winner': winner
            }
    except Exception as e:
        print(f'Resolution fetch notice: {e}')
        
    return {'open_price': None, 'close_price': None, 'completed': False, 'winner': None}

resolutions = {}
for slug, df_slot in slots_data.items():
    slot_start = int(df_slot['slot_start'].iloc[0])
    slot_end = int(df_slot['slot_end'].iloc[0])
    res = fetch_market_resolution(slot_start, slot_end)
    resolutions[slug] = res
    print(f'Slot {slug}: Winner = {res.get("winner")} (Open: {res.get("open_price")}, Close: {res.get("close_price")})')

Slot btc-updown-15m-1793942100: Winner = UP (Open: 75468.25068896638, Close: 75585.40578083936)
Slot btc-updown-5m-1793932800: Winner = DOWN (Open: 76056.14540208939, Close: 76003.73573112507)
Slot btc-updown-5m-1793933400: Winner = DOWN (Open: 76173.46488255, Close: 76102.52461971558)
Slot btc-updown-5m-1793940600: Winner = UP (Open: 75667.54824552496, Close: 75674.06358502647)
Slot btc-updown-5m-1793941200: Winner = DOWN (Open: 75658.0114113714, Close: 75578.32895759914)
Slot btc-updown-5m-1793941500: Winner = DOWN (Open: 75578.32895759914, Close: 75556.97308747382)
Slot btc-updown-5m-1793941800: Winner = DOWN (Open: 75556.97308747382, Close: 75468.25068896638)
Slot btc-updown-5m-1794454500: Winner = DOWN (Open: 73698.7547296367, Close: 73658.09786589454)


## 6. Strategy 1: Late-Entry Strategy

Enters late in the market window under low volatility with high directional confidence.

### Rules:
- **Entry Condition**:
  - `5s <= remaining_seconds <= 90s`
  - `atr_btc <= 2.0`
  - `gap_safety >= 40.0` ($|\text{gap}| / \text{ATR} \ge 40$)
  - `peak_gap_ratio >= 0.75`
  - Outcome ask $> 0.85$ with liquidity $\ge 20$
- **Stop-Loss Execution**:
  - Base threshold: `ask <= 0.48`
  - Suppressed if gap confirms ($|\text{gap}| > 5$) or RSI confirms momentum (`rsi >= 50` for UP, `rsi <= 50` for DOWN).
  - If `remaining < 20s`, RSI is ignored and stop executes if gap fails to confirm.

In [29]:
def run_late_entry_strategy(df: pd.DataFrame, resolution: Dict[str, Any], shares: float = 6.0) -> Dict[str, Any]:
    position = None
    trades = []
    
    for i, row in df.iterrows():
        remaining = row['remaining_seconds']
        gap = row['price_gap']
        atr = row.get('atr_btc', np.nan)
        gap_safety = row.get('gap_safety', np.nan)
        peak_gap_ratio = row.get('peak_gap_ratio', np.nan)
        rsi = row.get('rsi_gap', 50.0)
        
        up_best_ask = row.get('polymarket_up_best_ask', 1.0)
        down_best_ask = row.get('polymarket_down_best_ask', 1.0)
        up_asks = row.get('up_top5_asks', [])
        down_asks = row.get('down_top5_asks', [])
        up_bids = row.get('up_top5_bids', [])
        down_bids = row.get('down_top5_bids', [])
        
        # 1. Entry
        if position is None:
            if (
                5.0 <= remaining <= 90.0 and
                not pd.isna(atr) and atr <= 2.0 and
                not pd.isna(gap_safety) and gap_safety >= 40.0 and
                not pd.isna(peak_gap_ratio) and peak_gap_ratio >= 0.75
            ):
                up_eligible = (up_best_ask is not None and up_best_ask > 0.85 and row.get('polymarket_up_ask_liquidity', 0) >= 20)
                down_eligible = (down_best_ask is not None and down_best_ask > 0.85 and row.get('polymarket_down_ask_liquidity', 0) >= 20)
                
                if up_eligible or down_eligible:
                    side = 'UP' if up_eligible else 'DOWN'
                    target_asks = up_asks if side == 'UP' else down_asks
                    limit_price = up_best_ask if side == 'UP' else down_best_ask
                    
                    fill = L2OrderBookSimulator.simulate_buy(target_asks, shares_requested=shares, limit_price=limit_price)
                    if fill['status'] in ['FILLED', 'PARTIAL']:
                        position = {
                            'side': side,
                            'entry_time': row['current_time'],
                            'entry_price': fill['avg_fill_price'],
                            'shares': fill['filled_shares'],
                            'cost': fill['cost'],
                            'stop_loss_price': 0.48,
                        }
                        trades.append({
                            'type': 'BUY',
                            'side': side,
                            'time': row['current_time'],
                            'price': fill['avg_fill_price'],
                            'shares': fill['filled_shares'],
                            'cost': fill['cost'],
                            'reason': 'LATE_ENTRY_TRIGGER'
                        })
            continue
            
        # 2. Stop-loss
        if position is not None:
            side = position['side']
            target_best_ask = up_best_ask if side == 'UP' else down_best_ask
            target_bids = up_bids if side == 'UP' else down_bids
            
            gap_confirms = (gap > 5.0) if side == 'UP' else (gap < -5.0)
            rsi_confirms = (rsi >= 50.0) if side == 'UP' else (rsi <= 50.0)
            
            should_exit = False
            if 20.0 <= remaining <= 80.0:
                if target_best_ask is not None and target_best_ask <= position['stop_loss_price'] and not gap_confirms and not rsi_confirms:
                    should_exit = True
            elif remaining < 20.0:
                if target_best_ask is not None and target_best_ask <= position['stop_loss_price'] and not gap_confirms:
                    should_exit = True
                    
            if should_exit:
                sell_fill = L2OrderBookSimulator.simulate_sell(target_bids, shares_requested=position['shares'], limit_price=0.01)
                trades.append({
                    'type': 'SELL',
                    'side': side,
                    'time': row['current_time'],
                    'price': sell_fill['avg_fill_price'],
                    'shares': sell_fill['filled_shares'],
                    'proceeds': sell_fill['proceeds'],
                    'reason': 'STOP_LOSS_TRIGGER'
                })
                position = None
                break
                
    # Accounting
    total_cost = sum(t['cost'] for t in trades if t['type'] == 'BUY')
    total_proceeds = sum(t.get('proceeds', 0.0) for t in trades if t['type'] == 'SELL')
    
    terminal_payout = 0.0
    winner = resolution.get('winner')
    if position is not None and winner is not None:
        terminal_payout = position['shares'] * (1.00 if position['side'] == winner else 0.0)
        trades.append({'type': 'SETTLEMENT', 'side': position['side'], 'winner': winner, 'shares': position['shares'], 'payout': terminal_payout})
        
    net_pnl = total_proceeds + terminal_payout - total_cost
    return {
        'strategy': 'Late-Entry Strategy',
        'trades': trades,
        'total_cost': round(total_cost, 4),
        'total_proceeds': round(total_proceeds, 4),
        'terminal_payout': round(terminal_payout, 4),
        'net_pnl': round(net_pnl, 4),
        'outcome': 'WIN' if net_pnl > 0 else ('LOSS' if net_pnl < 0 else 'NO_TRADE')
    }

print('Late-Entry Strategy defined.')

Late-Entry Strategy defined.


## 7. Strategy 2: Limit Order Simulation Strategy

Baseline limit order execution strategy:
- Places a discount buy order at `0.49` early in the market window.
- Places a take-profit sell limit order at `0.70` upon fill.
- Emergency market sell if unfilled with 30 seconds remaining.

In [30]:
def run_simulation_limit_strategy(df: pd.DataFrame, resolution: Dict[str, Any], shares: float = 5.0) -> Dict[str, Any]:
    trades = []
    position = None
    take_profit_placed = False
    
    for i, row in df.iterrows():
        remaining = row['remaining_seconds']
        up_best_ask = row.get('polymarket_up_best_ask')
        up_best_bid = row.get('polymarket_up_best_bid')
        up_asks = row.get('up_top5_asks', [])
        up_bids = row.get('up_top5_bids', [])
        
        # Early buy at 0.49
        if position is None and not take_profit_placed and remaining > 100.0:
            if up_best_ask is not None and up_best_ask <= 0.49:
                fill = L2OrderBookSimulator.simulate_buy(up_asks, shares_requested=shares, limit_price=0.49)
                if fill['status'] in ['FILLED', 'PARTIAL']:
                    position = {
                        'side': 'UP',
                        'entry_time': row['current_time'],
                        'entry_price': fill['avg_fill_price'],
                        'shares': fill['filled_shares'],
                        'cost': fill['cost']
                    }
                    trades.append({
                        'type': 'BUY',
                        'side': 'UP',
                        'time': row['current_time'],
                        'price': fill['avg_fill_price'],
                        'shares': fill['filled_shares'],
                        'cost': fill['cost'],
                        'reason': 'DISCOUNT_LIMIT_BUY'
                    })
                    take_profit_placed = True
            continue
            
        # Monitor take-profit / emergency exit
        if position is not None:
            if up_best_bid is not None and up_best_bid >= 0.70:
                sell_fill = L2OrderBookSimulator.simulate_sell(up_bids, shares_requested=position['shares'], limit_price=0.70)
                trades.append({
                    'type': 'SELL',
                    'side': 'UP',
                    'time': row['current_time'],
                    'price': sell_fill['avg_fill_price'],
                    'shares': sell_fill['filled_shares'],
                    'proceeds': sell_fill['proceeds'],
                    'reason': 'TAKE_PROFIT_SELL'
                })
                position = None
                break
            elif remaining <= 30.0:
                sell_fill = L2OrderBookSimulator.simulate_sell(up_bids, shares_requested=position['shares'], limit_price=0.01)
                trades.append({
                    'type': 'SELL',
                    'side': 'UP',
                    'time': row['current_time'],
                    'price': sell_fill['avg_fill_price'],
                    'shares': sell_fill['filled_shares'],
                    'proceeds': sell_fill['proceeds'],
                    'reason': 'EMERGENCY_SELL_BEFORE_EXPIRY'
                })
                position = None
                break
                
    total_cost = sum(t['cost'] for t in trades if t['type'] == 'BUY')
    total_proceeds = sum(t.get('proceeds', 0.0) for t in trades if t['type'] == 'SELL')
    
    terminal_payout = 0.0
    winner = resolution.get('winner')
    if position is not None and winner is not None:
        terminal_payout = position['shares'] * (1.00 if position['side'] == winner else 0.0)
        trades.append({'type': 'SETTLEMENT', 'side': position['side'], 'winner': winner, 'payout': terminal_payout})
        
    net_pnl = total_proceeds + terminal_payout - total_cost
    return {
        'strategy': 'Limit Order Simulation Strategy',
        'trades': trades,
        'total_cost': round(total_cost, 4),
        'total_proceeds': round(total_proceeds, 4),
        'terminal_payout': round(terminal_payout, 4),
        'net_pnl': round(net_pnl, 4),
        'outcome': 'WIN' if net_pnl > 0 else ('LOSS' if net_pnl < 0 else 'NO_TRADE')
    }

print('Limit Order Simulation Strategy defined.')

Limit Order Simulation Strategy defined.


## 8. Multi-Slot Backtest Runner

Evaluates both strategies across all individual market slots.

In [31]:
def run_full_backtest(slots: Dict[str, pd.DataFrame], resolutions_map: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    records = []
    for slug, df_slot in slots.items():
        res = resolutions_map.get(slug, {})
        strat_runs = [
            run_late_entry_strategy(df_slot, res),
            run_simulation_limit_strategy(df_slot, res),
        ]
        for s in strat_runs:
            records.append({
                'market_slug': slug,
                'strategy': s['strategy'],
                'outcome': s['outcome'],
                'trades_count': len(s['trades']),
                'total_cost': s['total_cost'],
                'total_proceeds': s['total_proceeds'],
                'terminal_payout': s['terminal_payout'],
                'net_pnl': s['net_pnl'],
                'oracle_winner': res.get('winner', 'PENDING')
            })
    return pd.DataFrame(records)

backtest_df = run_full_backtest(slots_data, resolutions)
print('Backtest Execution Summary:')
display(backtest_df)

Backtest Execution Summary:


,market_slug,strategy,outcome,trades_count,total_cost,total_proceeds,terminal_payout,net_pnl,oracle_winner
0,btc-updown-15m-1793942100,Late-Entry Strategy,WIN,2,5.34,0.0,6.0,0.66,UP
1,btc-updown-15m-1793942100,Limit Order Simulation Strategy,NO_TRADE,0,0.00,0.0,0.0,0.00,UP
2,btc-updown-5m-1793932800,Late-Entry Strategy,NO_TRADE,0,0.00,0.0,0.0,0.00,DOWN
3,btc-updown-5m-1793932800,Limit Order Simulation Strategy,NO_TRADE,0,0.00,0.0,0.0,0.00,DOWN
4,btc-updown-5m-1793933400,Late-Entry Strategy,NO_TRADE,0,0.00,0.0,0.0,0.00,DOWN
5,btc-updown-5m-1793933400,Limit Order Simulation Strategy,LOSS,2,2.45,0.0,0.0,-2.45,DOWN
6,btc-updown-5m-1793940600,Late-Entry Strategy,NO_TRADE,0,0.00,0.0,0.0,0.00,UP
7,btc-updown-5m-1793940600,Limit Order Simulation Strategy,WIN,2,0.55,0.0,5.0,4.45,UP
8,btc-updown-5m-1793941200,Late-Entry Strategy,NO_TRADE,0,0.00,0.0,0.0,0.00,DOWN
9,btc-updown-5m-1793941200,Limit Order Simulation Strategy,NO_TRADE,0,0.00,0.0,0.0,0.00,DOWN


## 9. Performance Analytics & Strategy Comparison

In [32]:
summary_metrics = []
for strat_name, group in backtest_df.groupby('strategy'):
    total_trades = len(group[group['outcome'] != 'NO_TRADE'])
    wins = len(group[group['outcome'] == 'WIN'])
    losses = len(group[group['outcome'] == 'LOSS'])
    win_rate = (wins / total_trades * 100) if total_trades > 0 else 0.0
    
    total_pnl = group['net_pnl'].sum()
    gross_profits = group[group['net_pnl'] > 0]['net_pnl'].sum()
    gross_losses = abs(group[group['net_pnl'] < 0]['net_pnl'].sum())
    profit_factor = (gross_profits / gross_losses) if gross_losses > 0 else (np.inf if gross_profits > 0 else 0.0)
    
    summary_metrics.append({
        'Strategy': strat_name,
        'Slots Evaluated': len(group),
        'Active Trades': total_trades,
        'Wins': wins,
        'Losses': losses,
        'Win Rate (%)': round(win_rate, 2),
        'Total PnL ($)': round(total_pnl, 2),
        'Profit Factor': round(profit_factor, 2) if profit_factor != np.inf else 'Inf',
        'Avg PnL / Slot ($)': round(total_pnl / len(group), 2) if len(group) > 0 else 0.0
    })

df_summary = pd.DataFrame(summary_metrics)
print('Strategy Performance Comparison:')
display(df_summary)

Strategy Performance Comparison:


,Strategy,Slots Evaluated,Active Trades,Wins,Losses,Win Rate (%),Total PnL ($),Profit Factor,Avg PnL / Slot ($)
0,Late-Entry Strategy,8,1,1,0,100.0,0.66,Inf,0.08
1,Limit Order Simulation Strategy,8,4,1,3,25.0,-0.75,0.86,-0.09


## 10. Summary & Production Path

### Verification Highlights:
1. **Execution Reality**: Slippage against actual L2 book depth prevents false assumptions about fills.
2. **Late-Entry Discipline**: Strict criteria (`gap_safety >= 40.0`, `atr <= 2.0`, `ask > 0.85`) prevent trading during choppy noise.
3. **Stop-Loss / Emergency Exit**: Automatic expiry protection avoids holding dead positions into adverse resolutions.